# arg-position-back-functions — faded example 1: Complete maximum_back1 for out = maximum(x, y)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `arg-position-back-functions`. Running the beacon reports progress on the `Backprop: Arg-position back funcs` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Arg-position back funcs` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`arg-position-back-functions`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "arg-position-back-functions"
DD_SUBTOPIC = "Backprop: Arg-position back funcs"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Elementwise `out = maximum(x, y)` routes the gradient to whichever input was larger at each position. The two per-arg back fns are mask complements: `max_back0` passes `grad_out` where `x >= y`, `max_back1` passes `grad_out` where `y > x`. This is asymmetric per position and a classic place to get the mask direction wrong.

## Faded exercise 1

### Complete the per-arg back fns for `out = maximum(x, y)`

`max_back0` (gradient w.r.t. `x`) is given for you: the gradient flows to `x` exactly where `x >= y`. Complete `max_back1`, the gradient w.r.t. `y`, which must flow to `y` exactly where `y` strictly wins (`y > x`). Use the cached `x` and `y`; build a float mask and multiply by `grad_out`. (Tie-break: give the tie to arg 0, matching the `>=` above.)

**Fill in:** the gradient w.r.t. y: grad_out masked to the positions where y strictly exceeds x.

In [ ]:
def max_back0(grad_out, out, x, y):
    mask = (x >= y).to(grad_out.dtype)
    return grad_out * mask


def max_back1(grad_out, out, x, y):
    mask = (y > x).to(grad_out.dtype)
    return grad_out * mask


t.manual_seed(0)
x = t.randn(2, 4, requires_grad=True)
y = t.randn(2, 4, requires_grad=True)
out = t.maximum(x, y)
grad_out = t.randn(2, 4)
out.backward(grad_out)

gx = max_back0(grad_out, out.detach(), x.detach(), y.detach())
gy = max_back1(grad_out, out.detach(), x.detach(), y.detach())
print('grad_x match:', t.allclose(gx, x.grad))
print('grad_y match:', t.allclose(gy, y.grad))


def _test():
    t.manual_seed(0)
    x = t.randn(2, 4, requires_grad=True)
    y = t.randn(2, 4, requires_grad=True)
    out = t.maximum(x, y)
    grad_out = t.randn(2, 4)
    out.backward(grad_out)
    gy = max_back1(grad_out, out.detach(), x.detach(), y.detach())
    assert gy.shape == y.shape
    assert t.allclose(gy, y.grad), 'max_back1 must mask grad_out to where y > x'
    # independent reference
    ref = grad_out * (y.detach() > x.detach()).to(grad_out.dtype)
    assert t.allclose(gy, ref)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def max_back0(grad_out, out, x, y):
    mask = (x >= y).to(grad_out.dtype)
    return grad_out * mask


def max_back1(grad_out, out, x, y):
    mask = (y > x).to(grad_out.dtype)
    return grad_out * mask


t.manual_seed(0)
x = t.randn(2, 4, requires_grad=True)
y = t.randn(2, 4, requires_grad=True)
out = t.maximum(x, y)
grad_out = t.randn(2, 4)
out.backward(grad_out)

gx = max_back0(grad_out, out.detach(), x.detach(), y.detach())
gy = max_back1(grad_out, out.detach(), x.detach(), y.detach())
print('grad_x match:', t.allclose(gx, x.grad))
print('grad_y match:', t.allclose(gy, y.grad))
```
</details>